In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from collections import Counter
from tqdm import tqdm

# Check for MPS support
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

glove_file = 'glove.6B/glove.6B.100d.txt'
if not os.path.exists(glove_file):
    raise FileNotFoundError(f"{glove_file} not found. Please download it from https://nlp.stanford.edu/projects/glove/ and place it in your working directory.")

# -----------------------
# 1. Load Preprocessed Data
# -----------------------
df_train = pd.read_csv('train_preprocessed.csv')
df_test = pd.read_csv('test_preprocessed.csv')

# Ensure there are no NaN values in the text column
df_train['cleaned_text'] = df_train['cleaned_text'].fillna("")
df_test['cleaned_text'] = df_test['cleaned_text'].fillna("")

X_train = df_train['cleaned_text'].values
y_train = df_train['Bias'].values
X_test = df_test['cleaned_text'].values
y_test = df_test['Bias'].values

Using device: mps


In [2]:
# -----------------------
# 2. Encode Labels
# -----------------------
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
num_classes = len(le.classes_)
print("Number of classes:", num_classes)

Number of classes: 5


In [3]:

# -----------------------
# 3. Tokenization and Padding
# -----------------------

# Simple whitespace tokenization (you can substitute with a more robust tokenizer if needed)
def tokenize(text):
    return text.lower().split()

# Build vocabulary from training texts (limit vocabulary size)
max_words = 20000
counter = Counter()
for text in X_train:
    tokens = tokenize(text)
    counter.update(tokens)

# Reserve indices for PAD and OOV tokens
most_common = counter.most_common(max_words - 2)
word2idx = {"<PAD>": 0, "<OOV>": 1}
for word, count in most_common:
    word2idx[word] = len(word2idx)
vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)

# Convert texts to sequences of word indices
def text_to_sequence(text, word2idx):
    tokens = tokenize(text)
    return [word2idx.get(token, word2idx["<OOV>"]) for token in tokens]

X_train_seq = [text_to_sequence(text, word2idx) for text in X_train]
X_test_seq = [text_to_sequence(text, word2idx) for text in X_test]

# Pad sequences to a fixed length
max_seq_length = 100
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [word2idx["<PAD>"]] * (max_len - len(seq))
    else:
        return seq[:max_len]

X_train_pad = [pad_sequence(seq, max_seq_length) for seq in X_train_seq]
X_test_pad = [pad_sequence(seq, max_seq_length) for seq in X_test_seq]

# Convert lists to PyTorch tensors
X_train_tensor = torch.LongTensor(X_train_pad)
y_train_tensor = torch.LongTensor(y_train_encoded)
X_test_tensor = torch.LongTensor(X_test_pad)
y_test_tensor = torch.LongTensor(y_test_encoded)

Vocabulary size: 20000


In [4]:
# -----------------------
# 4. Create PyTorch Datasets and DataLoaders
# -----------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

train_dataset = TextDataset(X_train_tensor, y_train_tensor)
test_dataset = TextDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [5]:
# -----------------------
# 5. Load GloVe Embeddings and Build Embedding Matrix
# -----------------------
embedding_dim = 100  # Using GloVe 100d embeddings

embeddings_index = {}
with open(glove_file, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector
print("Loaded %d word vectors from GloVe." % len(embeddings_index))

# Build embedding matrix: for each word in our vocabulary, use the GloVe vector if available
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

# Convert embedding matrix to PyTorch tensor
embedding_matrix_tensor = torch.FloatTensor(embedding_matrix)

Loaded 400000 word vectors from GloVe.


In [7]:
# -----------------------
# 6. Define the LSTM Model in PyTorch
# -----------------------
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix, num_layers=1, dropout=0.5):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = False  # Freeze embeddings; set True to fine-tune
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        embedded = self.embedding(x)  # [batch, seq_len, embedding_dim]
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use the last hidden state (from the last layer)
        hidden = self.dropout(hidden[-1])
        out = self.fc(hidden)
        return out

hidden_dim = 128
num_layers = 1
dropout = 0.5
model = LSTMClassifier(vocab_size=vocab_size,
                       embedding_dim=embedding_dim,
                       hidden_dim=hidden_dim,
                       output_dim=num_classes,
                       embedding_matrix=embedding_matrix_tensor,
                       num_layers=num_layers,
                       dropout=dropout)
model = model.to(device)
print(model)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

LSTMClassifier(
  (embedding): Embedding(20000, 100)
  (lstm): LSTM(100, 128, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=128, out_features=5, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


In [8]:
# -----------------------
# 7. Train the Model
# -----------------------
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    epoch_correct = 0
    total = 0
    for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * texts.size(0)
        _, predicted = torch.max(outputs, 1)
        epoch_correct += (predicted == labels).sum().item()
        total += labels.size(0)
    epoch_acc = epoch_correct / total
    print(f"Epoch {epoch+1} Loss: {epoch_loss/total:.4f} Acc: {epoch_acc:.4f}")

Epoch 1/10: 100%|██████████| 56/56 [00:00<00:00, 66.00it/s]


Epoch 1 Loss: 1.4160 Acc: 0.4778


Epoch 2/10: 100%|██████████| 56/56 [00:00<00:00, 58.96it/s]


Epoch 2 Loss: 1.3239 Acc: 0.5171


Epoch 3/10: 100%|██████████| 56/56 [00:00<00:00, 63.87it/s]


...

Epoch 8 Loss: 1.1461 Acc: 0.5571


Epoch 9/10: 100%|██████████| 56/56 [00:00<00:00, 62.28it/s]


Epoch 9 Loss: 1.1015 Acc: 0.5705


Epoch 10/10: 100%|██████████| 56/56 [00:00<00:00, 63.80it/s]

Epoch 10 Loss: 1.0403 Acc: 0.6009


In [9]:
# -----------------------
# 8. Evaluate the Model on the Test Set
# -----------------------
model.eval()
test_loss = 0
correct = 0
total = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = model(texts)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * texts.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= total
test_acc = correct / total
print(f"Test Loss: {test_loss:.4f} Test Accuracy: {test_acc:.4f}")

# Print classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

Test Loss: 1.4631 Test Accuracy: 0.4135
Classification Report:
              precision    recall  f1-score   support

      center       0.17      0.10      0.13        40
   lean left       0.07      0.05      0.06        61
  lean right       0.00      0.00      0.00        37
        left       0.53      0.68      0.59       230
       right       0.30      0.27      0.28        77

    accuracy                           0.41       445
   macro avg       0.21      0.22      0.21       445
weighted avg       0.35      0.41      0.38       445

